In [1]:
import os, sys
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(project_root)

d:\myproject


In [2]:
from backend.parser_bridge import parse_resume_source, parse_job_source, derive_parser_mode
from backend.ats_score_engine import score_resume_against_job
from backend.resume_feedback_engine import generate_candidate_feedback, generate_internal_feedback

In [3]:
resume_text = '''Python developer with 3 years of experience in machine learning, SQL, data analysis, and NLP projects. Built classification models and analytics dashboards using pandas, scikit-learn, and FastAPI.'''

job_text = '''We are hiring a data scientist with Python, SQL, machine learning, NLP, and model building experience. Candidates with analytics dashboard exposure and production deployment experience will be preferred.'''

prefer_gemini = True

In [4]:
resume_bundle = parse_resume_source(raw_text=resume_text, prefer_gemini=prefer_gemini)
job_bundle = parse_job_source(job_text, prefer_gemini=prefer_gemini)
parser_mode = derive_parser_mode(resume_bundle['source'], job_bundle['source'])

result = score_resume_against_job(
    parsed_resume=resume_bundle['parsed'],
    parsed_job=job_bundle['parsed'],
    resume_raw_text=resume_bundle['text'],
    job_raw_text=job_bundle['text'],
    parser_mode=parser_mode,
    parser_sources={'resume': resume_bundle['source'], 'job': job_bundle['source']},
)

internal = generate_internal_feedback(resume_bundle['parsed'], job_bundle['parsed'], result)
candidate = generate_candidate_feedback(resume_bundle['parsed'], job_bundle['parsed'], result)

print('Parser mode:', parser_mode)
print('Resume parser source:', resume_bundle['source'])
print('Job parser source:', job_bundle['source'])
print('Warnings:', resume_bundle['warnings'] + job_bundle['warnings'])

d:\myproject\myproject\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4856.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parser mode: parser_enhanced_gemini
Resume parser source: local_structured_fallback
Job parser source: gemini_job_parser
Warnings: ['Resume parser failed via parse_resume_file: expected str, bytes or os.PathLike object, not NoneType']


d:\myproject\myproject\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [5]:
result.model_dump() if hasattr(result, 'model_dump') else result.dict()

{'legacy_ats_score': 55.83,
 'ml_score': 67.21,
 'final_hybrid_score': 62.66,
 'ats_score': 52.77,
 'ranking_score': 56.6,
 'resume_quality_score': 40.69,
 'job_match_score': 62.66,
 'combined_score': 52.77,
 'model_ranking_score': 56.6,
 'score_summary': {'score_version': 'v3_split_quality_match',
  'resume_quality_score': 40.69,
  'job_match_score': 62.66,
  'combined_score': 52.77,
  'ranking_score': 56.6,
  'legacy_ats_score': 55.83,
  'ml_score': 67.21,
  'model_ranking_score': 56.6},
 'match_label': 'strong_match',
 'model_name': 'Random Forest',
 'parser_mode': 'parser_enhanced_gemini',
 'parser_sources': {'resume': 'local_structured_fallback',
  'job': 'gemini_job_parser'},
 'class_probabilities': {'poor_match': 0.00963,
  'moderate_match': 0.318231,
  'strong_match': 0.672138},
 'matched_required_skills': ['machine learning', 'python', 'sql'],
 'missing_required_skills': ['natural language processing'],
 'matched_preferred_skills': [],
 'missing_preferred_skills': [],
 'matche

In [6]:
candidate.model_dump() if hasattr(candidate, 'model_dump') else candidate.dict()

{'summary': 'Your base resume quality score is 40.69, this job match score is 62.66, and the combined screening score is 52.77.',
 'match_label': 'strong_match',
 'ats_score': 52.77,
 'resume_quality_score': 40.69,
 'job_match_score': 62.66,
 'combined_score': 52.77,
 'strengths': ['You already match these important skills: machine learning, python, sql',
  'Your experience level appears well aligned with the role.'],
 'improvement_areas': ['Improve overall ATS readability by making skills, experience, and education easier to detect.',
  'Add stronger project bullets with outcomes, tools, and measurable impact.',
  'Add evidence for these missing required skills if you truly have them: natural language processing',
  'Include more role-specific keywords such as: natural language processing, we, are, hiring, scientist, building, candidat, exposure'],
 'general_improvement_areas': ['Improve overall ATS readability by making skills, experience, and education easier to detect.',
  'Add str

In [7]:
internal.model_dump() if hasattr(internal, 'model_dump') else internal.dict()

{'summary': 'Candidate classified as strong_match. Resume quality is 40.69, job match is 62.66, and combined score is 52.77. Legacy score is 55.83, ML score is 67.21, and parser mode is parser_enhanced_gemini.',
 'match_label': 'strong_match',
 'ats_score': 52.77,
 'resume_quality_score': 40.69,
 'job_match_score': 62.66,
 'combined_score': 52.77,
 'recommendation': 'Do not prioritize unless the role is flexible or the candidate pool is limited.',
 'decision_bucket': 'reject',
 'strengths': ['Matched required skills: machine learning, python, sql',
  'Experience alignment appears strong.',
  'Model confidence for strong-match class is high.'],
 'risks': ['Missing required skills: natural language processing',
  'Important role keywords are underrepresented in the resume.',
  'Resume quality is low enough that formatting or evidence clarity may reduce ATS performance.'],
 'accept_reasons': ['Required skill evidence found for: machine learning, python, sql',
  'Experience evidence aligns